Default authentication
The Databricks SDK exposes a Workspace Client class that you will use throughout this course to interact with a Databricks workspace. By default, this class is instantiated without parameters, as the SDK automatically retrieves the environment variables needed to authenticate to a Databricks workspace.

The variable dbrx_host, containing the address of the Databricks host, has been preloaded.

Import the class from the Databricks SDK that contains the Workspace Client.
Use the os library to set the authentication environment variable to the predefined dbrx_host.
Create a Workspace Client object using default authentication.

In [0]:
import os
# Import the workspace client
from databricks.sdk import WorkspaceClient

# Set the Databricks authentication environment variables
os.environ['DATABRICKS_HOST'] = dbrx_host

# Instantiate the Workspace Client
workspace_client = WorkspaceClient()

Listing Databricks jobs
Now, we will list the Databricks jobs that have been created in our workspace. The WorkspaceClient jobs API exposes a .list() method that can be used to list all jobs in the authenticated workspace.

Listing Databricks jobs
Now, we will list the Databricks jobs that have been created in our workspace. The WorkspaceClient jobs API exposes a .list() method that can be used to list all jobs in the authenticated workspace.

Retrieve all jobs in the workspace using the WorkspaceClient.
Iterate through up to 10 jobs and print each job's job_id attribute.

Retrieve all jobs in the workspace using the WorkspaceClient.
Iterate through up to 10 jobs and print each job's job_id attribute.

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
# Retrieve all jobs in the workspace using the `WorkspaceClient`
jobs = w.jobs.list()
# Iterate through up to 10 jobs in workspace and print the `job_id`
for job in list(jobs)[:10]:
    print(f"JobId={job.job_id}")

Listing Databricks clusters
Databricks clusters are a set of virtual computers hosted in the Databricks platform that allow you to run Databricks jobs and notebooks. Similar to the jobs API, the clusters API of the WorkspaceClient exposes a .list() function that lists all clusters in the authenticated workspace.

Iterate through all clusters.
Print the cluster id for each cluster.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Iterate through all clusters
for c in w.clusters.list():
  # Print cluster id for each cluster
  print(c.cluster_id)

Query a Foundation Model
Let's use what we learned about the difference between "SYSTEM" and "USER" Chat Message Roles to send different prompts to the AI agent based on the Chat Message role.

Tell the AI Agent to act like a high school computer science teacher when answering queries using the prompt ai_agent_context.
Ask the AI agent when to use For loops using the user_prompt.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
w = WorkspaceClient()
user_prompt = "What is a For loop and When should I use it?"
ai_agent_context = "You are a high school computer science teacher"
response = w.serving_endpoints.query(
    name="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        ChatMessage( 
          	# Set system prompt
            role=ChatMessageRole.SYSTEM, content=ai_agent_context
        ),
        ChatMessage( 
          	# Set user query
            role=ChatMessageRole.USER, content=user_prompt
        ),
    ],
    max_tokens=250)
print(f"RESPONSE:\n{response.choices[0].message.content}")

Query chatbot using Databricks Servings Endpoint API
In the previous exercise, you learned how to use the Databricks SDK to query an AI Foundation model. The goal of this exercise is for you to experiment with different user and system prompts when querying AI models using the Databricks SDK

Edit the System prompt and see how the AI model responds differently.
Edit the User prompt and see how the AI model responds differently.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
w = WorkspaceClient()
response = w.serving_endpoints.query(
    name="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        ChatMessage( 
          	# Edit the System prompt
            role=ChatMessageRole.SYSTEM, content="You are good python programmer for developer"
        ),
        ChatMessage( 
          	# Edit the User prompt
            role=ChatMessageRole.USER, content="How Python loop works in SDK?" 
        ),
    ],
    max_tokens=128)
print(f"RESPONSE:\n{response.choices[0].message.content}")

Text summarization
Want to turn a poem into a quick summary? Let’s make it happen! You have the full text of a poem stored as a string variable called raven_text, which you will hand off to a Chat AI model and ask it to summarize the poem in a set number of words.

Ask the AI agent to summarize the poem "The Raven" by Edgar Allan Poe.
Define the model name as Meta Llama 3, already saved as ai_model.
Create the message you want to send to the AI model, including the raven_text variable.
Use the chat message role used to query the AI model.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

ai_model = "databricks-meta-llama-3-3-70b-instruct"

raven_text = "The Raven by Edgar Allan Poe"

response = w.serving_endpoints.query(
    name=ai_model,
    messages=[
        ChatMessage(
            role=ChatMessageRole.USER,
            content=f"Summarize this poem by Edgar Allan Poe: {raven_text} in 3 bullet points"
        )
    ],
    max_tokens=200
)

print(f"RESPONSE:\n{response.choices[0].message.content}")

Proofreading text
Have text that you want to proofread? Use the Databricks SDK to ask an AI model to proofread your text for you. It will send you back the proofread version of your text and a list of the specific changes it made.

Query the class in the workspace client used to query AI models.
Fill in the role used to provide instructions to the AI model on how to answer future queries.
Fill in the role used when sending a query to the AI model.
Print the message details from the AI model response.

In [0]:
w = WorkspaceClient()

text_to_proofread = "I am gong to the stor to buy some bred. I migt also buy some mlk. Do u want to jon me?"

# Query the AI model via the workspace client
response = w.serving_endpoints.query(
    name="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        ChatMessage(
            # Specify the role for configuring model behavior
            role=ChatMessageRole.SYSTEM,
            content="You are a helpful writing assistant."
        ),
        ChatMessage(
            # Specify the role for sending the query
            role=ChatMessageRole.USER,
            content=f"Proofread the following text: {text_to_proofread}"
        ),
    ],
    max_tokens=200
)

# Print the content of the model's response
print(f"RESPONSE:\n{response.choices[0].message.content}")

Create LangChain SQL Agent Connected to Databricks
LangChain is a library to build and deploy LLM applications. It also allows you to create AI agents, connecting connect to data sources and running SQL queries.

In the following exercises, you will be interacting with schemas from the samples catalog. In this exercise, you will create a Databricks SQL agent and in subsequent exercises you will also query the agents that you create.

We import the following functions into the coding environment for you:

from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase
from databricks_langchain import ChatDatabricks
from databricks.sdk import WorkspaceClient

Create a LangChain SQLDatabase from the nyctaxi schema in the "samples" Databricks catalog.
Create a Databricks LangChain LLM using the Databricks Meta Llama 3 Foundational Model.
Create a LangChain SQL Database toolkit that uses the LangChain Databricks SQL db and the llm.
Create a LangChain Databricks SQL agent using the toolkit and llm.

In [0]:
# Connect to the nyctaxi schema
db = SQLDatabase.from_databricks(
    catalog="samples",
    schema="nyctaxi",
    warehouse_id=warehouse_id
)

# Create the LangChain LLM
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    max_tokens=100
)

# Build the SQL toolkit
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Create the SQL agent
databricks_sql_agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    handle_parsing_errors=True
)

Query NYC Taxi Dataset using SQL Agent
Databricks exposes an open source samples catalog that contains several schemas. In this exercise, we will create a LangChain Databricks SQL agent to connect to the nyctaxi schema and then ask it a question that requires SQL to answer. Creating one of these agents empowers the user to perform data analysis and ask questions on data that require SQL, with no prior knowledge of SQL.

Create LangChain SQLDatabase object from nyctaxi schema in Databricks samples catalog.
Create a custom LangChain LLM using the Databricks Meta Llama 3 model.
Create a LangChain Databricks SQL Agent using the custom LLM.
Query the Databricks SQL Agent.

In [0]:
# Connect to the nyctaxi schema
db = SQLDatabase.from_databricks(
    catalog="samples", 
    schema="nyctaxi",  
  	warehouse_id=warehouse_id)

# Create the custom LangChain LLM
meta_llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    max_tokens=100)
toolkit = SQLDatabaseToolkit(db=db, llm=meta_llm)

# Create the SQL agent
agent = create_sql_agent(llm=meta_llm, toolkit=toolkit, verbose=True, handle_parsing_errors=True)

# Query the Databricks SQL Agent
result = agent.run("what's the duration and distance of the longest trip?")
print(result)

Query bakery sales data using SQL agent
The "bakehouse" schema in the samples catalog in Databricks contains fabricated data about bakery sales. We can create a LangChain Databricks SQL agent to connect to this dataset and then ask the agent queries about the data. The agent can use SQL to get the answers and can show it's work, describing the SQL queries and logic used to get to the solution.

Create a LangChain SQL database from the bakehouse dataset in the samples catalog.
Create a SQLDatabaseToolkit object that connects to the LangChain SQL database and llm.
Create a LangChain SQL Agent that connects to the llm and toolkit.
Ask the Databricks SQL agent which product generates the most revenue.

In [0]:
# Connect to the bakehouse schema
bakehouse_db = SQLDatabase.from_databricks(
    catalog="samples", 
    schema="bakehouse",  
    warehouse_id=warehouse_id)

# Build the SQL toolkit
toolkit = SQLDatabaseToolkit(db=bakehouse_db, llm=llm)
# Create the SQL agent
agent = create_sql_agent(llm=llm, toolkit=toolkit, verbose=True, handle_parsing_errors=True)

# Run the revenue query
prompt = "Which product generates the most revenue?"
result = agent.run(prompt)
print(result)

Start a cluster
Clusters oftentimes are configured to turn off and run in IDLE mode after being inactive for a certain period of time. We can use the clusters service of the Databricks SDK to run a cluster that exists in the authenticated workspace.

Store value of environment variable DATACAMP_CLUSTER_ID in a variable called cluster_id.
Start cluster with id stored in environment variable DATACAMP_CLUSTER_ID.

In [0]:
w = WorkspaceClient()

# Store value of environment variable as cluster_id
cluster_id = os.environ['DATACAMP_CLUSTER_ID']

# Start cluster with id stored in cluster_id variable
try:
    w.clusters.start(cluster_id=cluster_id).result()
except:
    print(f"Cannot start cluster_id={cluster_id} because it is already running")

Check that a cluster is running
We can use the clusters service in the Databricks SDK to retrieve information about a specified cluster. In this exercise, we are interested in looking at the state of a cluster. This informs us whether a cluster is Terminated, Idle or in a Running state. Assume that the id of the cluster we are interested in is stored in a variable called cluster_id.

Retrieve cluster information for cluster with id stored in variable cluster_id.
Print the state of cluster to confirm it is running.

In [0]:
w = WorkspaceClient()

# Retrieve info for the specified cluster
cluster_info = w.clusters.get(cluster_id)

# Print the state of the cluster
print(cluster_info.state)

Create and run a Databricks job
In this exercise, you will create a job that runs a notebook with name HelloWorld that simply prints "Hello World". You can run whatever code that you want in the notebook referenced in the job.

This job needs to be explicitly told when to run.

The notebook path is dependent on the username of the authenticated user for the workspace and is structured as follows: /Users/${username}/${notebook_name}.

We have already created the WorkspaceClient(), storing as w.

Retrieve the username of the current user in the workspace to generate the notebook path for notebook with name HelloWorld.
Create a job that runs the HelloWorld notebook.
Run the job you just created.

In [0]:
# Build notebook path for HelloWorld
notebook_path = f'/Users/{w.current_user.me().user_name}/HelloWorld'

# Create a job that runs the HelloWorld Notebook
created_job = w.jobs.create(
    name='sdk-dc-project-task',
    tasks=[
        jobs.Task(
            description="create_notebook_test",
            # Set the notebook_path
            notebook_task=jobs.NotebookTask(
                notebook_path=notebook_path
            ),
            task_key="my-key"
        )
    ]
)

# Run job
w.jobs.run_now(created_job.job_id)

Schedule a Databricks job
In this exercise, you will use the Databricks Workspace Client to create a job that runs a notebook with name HelloWorld at a scheduled time.

You already learned how to create a job that can be explicitly triggered. By passing in the schedule parameter to this function, you can schedule the job to run at a specific time by passing in a cron expression.

In this exercise, you can assume there is a predefined variable called cron_expression that represent the cron expression to run once a day at 3 am Eastern Time, respectively.

We have already created the WorkspaceClient(), storing as w, and a notebook_path variable.

Use the create() method in the WorkspaceClient.jobs API to create a job.
Schedule the job being created to run every day at 3 am in the eastern timezone, using the preset cron_expression variable.
Configure the job to time out after 25 seconds.

In [0]:
# Create a job scheduled to run every day at 3 AM
created_job = w.jobs.create(
    name='sdk-dc-project-task',
    tasks=[
        jobs.Task(
            description="test",
            notebook_task=jobs.NotebookTask(
                notebook_path=notebook_path
            ),
            task_key="my-key"
        )
    ],
    # Schedule it to run every day at 3 AM
    schedule=jobs.CronSchedule(
        quartz_cron_expression="0 0 3 * * ?",
        timezone_id="America/New_York"
    ),

    # The job should timeout after 25 seconds
    timeout_seconds=25,
)

Delete a Databricks job
You learned how to create and run a Databricks job using the Python SDK. Sometimes there will be jobs that you previously created that are now deprecated so you want to delete them. You can use the SDK to programatically delete these jobs instead of going into the Databricks console to manually delete them. In this exercise, you will create a job and then use the SDK to delete that job.

We have already created the WorkspaceClient(), storing as w, and a notebook_path variable.

Create a Databricks job using the WorkspaceClient that runs on cluster_id.
Delete the Databricks job created in step 1.

In [0]:
# Create a job with the notebook task
created_job = w.jobs.create(
    name='sdk-dc-project-task',
    tasks=[
        jobs.Task(
            description="create_notebook_test",
            existing_cluster_id=cluster_id,
            notebook_task=jobs.NotebookTask(
                notebook_path=notebook_path
            ),
            task_key="my-key"
        )
    ]
)

# Delete the job that was just created
w.jobs.delete(job_id=created_job.job_id)